# Module 01: Data Engineering Foundations

**Program:** Quintrix Jr. Data Engineer Training  
**Duration:** 3 hours  
**Instructor:** Sunil Mogadati

---

**Today's Agenda:**

| # | Topic | Time |
|---|-------|------|
| 1 | Data Engineering Role and Responsibilities | 35 min |
| 2 | Data Lifecycle: Generation → Storage → Processing → Serving | 40 min |
| 3 | ETL vs. ELT Paradigms | 40 min |
| 4 | Batch vs. Real-Time Processing | 30 min |
| 5 | Modern Data Stack Overview | 35 min |

In [ ]:
# ============================================================# HELLO WORLD — Your First Data Pipeline# ============================================================# This is a pipeline: data in → transform → data out.# Every DE job follows this pattern, from a 3-line script to a# 10,000-line PySpark pipeline running on 100 machines.# STEP 1: DATA IN — raw records (list of dicts = most common raw format)raw_records = [    {"name": "Alice", "department": "Engineering", "salary": 95000},    {"name": "Bob",   "department": "Marketing",   "salary": 72000},    {"name": "Carol", "department": "Engineering", "salary": 110000},    {"name": "Dave",  "department": "Marketing",   "salary": 68000},    {"name": "Eve",   "department": "Engineering", "salary": 102000},]# STEP 2: TRANSFORM — filter to one department# WHY: In production, you rarely want ALL data. You filter, clean, enrich.engineers = [r for r in raw_records if r["department"] == "Engineering"]# STEP 3: DATA OUT — print the result (in production: write to database/file)# BEGINNER NOTE: This tiny script IS a pipeline. The shape never changes:#   read → transform → write. Everything else is just scale.print("=== My First Data Pipeline ===")print(f"Input:  {len(raw_records)} records")print(f"Output: {len(engineers)} engineers")for eng in engineers:    print(f"  {eng['name']}: ${eng['salary']:,}")# You should see:#   Input:  5 records#   Output: 3 engineers#   Alice: $95,000#   Carol: $110,000#   Eve: $102,000

---

## 1. Data Engineering Role and Responsibilities

**One-line answer:** Data engineers build the systems that move data from where it's generated to where it's useful.

You don't analyze data. You don't build dashboards. You don't train models. You build the **infrastructure** that makes all of that possible.

> **Analogy:** If data scientists are chefs, data engineers build the kitchen — the plumbing, the gas lines, the refrigeration, the delivery trucks. No kitchen, no cooking.

### What Data Engineers Actually Do

| Activity | Example |
|----------|----------|
| **Build data pipelines** | Write PySpark jobs that transform raw call logs into clean, queryable tables |
| **Design data models** | Create star schemas so analysts can answer business questions with simple JOINs |
| **Move data between systems** | Extract from MongoDB, transform with Spark, load into BigQuery |
| **Ensure data quality** | Catch duplicates, validate schemas, mask PII before data reaches reports |
| **Orchestrate workflows** | Schedule Airflow DAGs that run pipelines daily at 6 AM |
| **Optimize performance** | Fix a Spark job that takes 4 hours to run in 20 minutes |
| **Monitor and troubleshoot** | Get an alert at 3 AM that yesterday's data didn't load, diagnose and fix |
| **Maintain infrastructure** | Manage cloud resources, databases, clusters, storage |

### How Data Engineers Differ from Other Roles

| | Data Engineer | Data Analyst | Data Scientist | ML Engineer |
|---|---|---|---|---|
| **Primary question** | "How do we get the data there?" | "What does the data say?" | "Can we predict/model this?" | "How do we deploy this model?" |
| **Main tools** | Spark, Airflow, SQL, Cloud | SQL, Tableau, Excel | Python, R, Jupyter | PyTorch, MLflow, Docker |
| **Output** | Pipelines, tables, APIs | Reports, dashboards | Models, insights | Production model services |
| **Closest analogy** | Plumber / Electrician | Accountant | Research scientist | Factory engineer |
| **Thinks about** | Throughput, latency, reliability | Trends, KPIs, anomalies | Accuracy, features, bias | Inference speed, scaling |

> **Key insight:** In many companies, especially startups and mid-size firms, these roles blur. You'll meet "Analytics Engineers" (DE + DA) and "ML Engineers" who also build their own pipelines. Knowing DE makes you more valuable in ANY data role.

### Day in the Life of a Jr. Data Engineer

**9:00 AM** — Check Airflow dashboard. Last night's daily pipeline ran successfully. One warning: a source file had 3% more null values than usual.

**9:30 AM** — Standup. PM asks: "Can we add a new column to the daily report — average call duration by campaign?" You say: "Yes, I can add it to the Silver layer transform and the BigQuery mart. Probably 2 hours."

**10:00 AM** — Write a PySpark transformation to compute avg call duration grouped by campaign. Add it to the existing pipeline.

**11:30 AM** — Write a data quality test: assert avg_duration is between 30 and 1800 seconds. If it's outside that range, something is wrong with the source data.

**1:00 PM** — Code review from senior DE. They suggest using a window function instead of a group-by + join. You learn something.

**2:00 PM** — Deploy the updated pipeline via CI/CD (GitHub Actions). It runs in the staging environment first.

**3:00 PM** — Investigate a data quality alert from yesterday. An upstream system changed their date format from `MM/DD/YYYY` to `YYYY-MM-DD`. Your pipeline broke. You add a format-detection step.

**4:30 PM** — Document the schema change in the data dictionary. Update the runbook.

> **Notice:** Most of the day is writing Python/SQL, debugging data issues, and working with other engineers. It's not abstract — it's very hands-on.

---

## 2. Data Lifecycle: Generation → Storage → Processing → Serving

**One-line answer:** Every piece of data flows through four stages — generation, storage, processing, serving — and your job as a DE covers the last three.


Every piece of data in every company follows the same journey:

```
GENERATION  →  STORAGE  →  PROCESSING  →  SERVING
(something     (put it      (clean it,     (make it
 happens)      somewhere)    transform)     useful)
```

| Stage | What Happens | Example (Call Center) |
|-------|-------------|----------------------|
| **Generation** | An event occurs that produces data | A customer calls and the AI agent records the call |
| **Storage** | Raw data lands somewhere | Call record is saved as JSON in MongoDB |
| **Processing** | Data is cleaned, validated, transformed, enriched | Deduplicate calls, convert UTC→EST, join call to order |
| **Serving** | Processed data is made available for consumption | Daily campaign report shows conversion rates in BigQuery |

> **Your job as a DE** is stages 2, 3, and 4. You don't control what generates the data (that's the application team), but everything after that is yours.

---

## Hello World: Your First Data Pipeline

Before we look at a real call-center event, run this short pipeline. Every pipeline in this program — however large — is this same idea scaled up.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HELLO WORLD: Your First Data Pipeline
#
# Before we look at a real call-center event, let's build the smallest possible
# pipeline in ~15 lines. Every production pipeline in this program — no matter
# how large — is this same concept repeated and scaled.
#
# Pipeline = a sequence of steps that transforms data from one form to another
#   DATA IN  ->  CLEAN / TRANSFORM  ->  DATA OUT
# ─────────────────────────────────────────────────────────────────────────────

# Step 1: DATA IN — imagine this came from a CSV file, an API (Application Programming Interface),
# or a database table. CSV = Comma-Separated Values — the most common flat-file data format.
# BEGINNER NOTE: In Python, a string that starts and ends with triple quotes can span many lines.
raw_csv = (
    "name,score\n"
    "Alice,95\n"
    "Bob,\n"    "Carol,88\n"
    "Dave,72\n"
)
# WHY intentional null: real datasets always have missing values.
# A pipeline must handle them gracefully rather than crash.

# Step 2: Parse — split the text into rows and columns
lines   = raw_csv.strip().split('\n')   # strip() removes leading/trailing whitespace
header  = lines[0].split(',')           # first line = column names
records = [line.split(',') for line in lines[1:]]  # remaining lines = data rows
# list comprehension = compact way to build a list: [expression for item in iterable]

# Step 3: TRANSFORM — clean nulls (remove any record missing a score)
# WHY: nulls in numeric fields cause averages and sums to silently produce wrong results
cleaned = []
for row in records:
    name, score = row[0], row[1]        # unpack two columns from each row
    if score.strip():                   # WHY .strip(): '  ' (spaces only) is falsy when stripped
        cleaned.append({'name': name, 'score': int(score)})  # int() converts '95' -> 95
    else:
        print(f"  [SKIPPED] '{name}' has a null score — dropped from pipeline")

# Step 4: DATA OUT — result ready for analysis or loading into a database
print()
print('Pipeline complete. Clean records:')
for r in cleaned:
    print(f"  {r['name']:8s}  score={r['score']}")

print()
print('This IS a pipeline: data in -> transform -> data out.')
print('Everything in this program scales this concept:')
print('  Larger data   -> PySpark (Modules 06-07)')
print('  Cloud storage -> GCS / S3 (Module 08)')
print('  Scheduling    -> Apache Airflow = workflow orchestration tool (Module 11)')
# You should see: Bob is skipped (null score), Alice/Carol/Dave printed with integer scores


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# The Data Lifecycle in Python — a simple mental model
# DE = Data Engineer | Your job covers Storage -> Processing -> Serving
# This single record flows through all 4 lifecycle stages shown in this section.
# ─────────────────────────────────────────────────────────────────────────────

# Stage 1: GENERATION — something happens in the real world
# WHY: We start with a raw event because that is what the source system produces.
# A VA (Virtual Assistant) platform records every call as a JSON object.
# JSON = JavaScript Object Notation — a common text format for exchanging data between systems.
raw_event = {
    "call_id": "call_ac567d48edabbd24be4eaad",  # unique identifier for this call
    "dnis": "8005551234",           # DNIS = Dialed Number Identification Service (the number the caller dialed)
    "caller_phone": "3135559876",   # caller's phone number
    "start_time": "2026-03-10T19:30:00Z",   # UTC timestamp — "Z" means Zulu = UTC = no timezone offset
    "end_time": "2026-03-10T19:37:45Z",
    "disposition": "completed",     # outcome of the call (completed, dropped, transferred)
    "customer": {
        # nested dict — customer info lives inside the call record
        "first_name": "Jane",
        "last_name": "Smith",
        "city": "Detroit",
        "state": "MI"
    }
}

# BEGINNER NOTE: A Python dict (dictionary) is a collection of key-value pairs.
# You access a value with: raw_event["call_id"] -> "call_ac567d48edabbd24be4eaad"

print("Stage 1 — GENERATION: Raw event from the VA platform")
print(f"  Call ID: {raw_event['call_id']}")
print(f"  Raw timestamp: {raw_event['start_time']} (UTC)")
print(f"  Customer nested inside: {raw_event['customer']}")
print()
# You should see: the call_id, a UTC timestamp ending in Z, and a nested dict for customer


In [ ]:
import json  # json = standard Python library for encoding/decoding JSON (JavaScript Object Notation)

# Stage 2: STORAGE — save the raw data as-is (Bronze layer)
# WHY: Raw data must be stored untouched so you can always replay or audit it.
# If your transform code has a bug, you need the original data to fix it.
# Bronze layer = first landing zone — raw, immutable (never changed after arrival).
# In production: GCS (Google Cloud Storage), S3 (Amazon Simple Storage Service),
# or a Delta Lake table on cloud object storage.
# Here: simulate by serialising to a JSON string.

# json.dumps = JSON encode (dict -> string). indent=2 = pretty-print with 2-space indentation.
raw_json = json.dumps(raw_event, indent=2)

print("Stage 2 — STORAGE: Raw JSON saved exactly as received")
print(raw_json)
print()

# WHY this rule matters:
# Real incident: a transform dropped a column. Because we kept raw data in Bronze,
# we re-ran the transform and recovered. Without Bronze, that data was gone forever.
print("Rule: NEVER modify raw data. Store it as-is. You might need it later.")
# You should see: the same raw_event dict printed as formatted JSON text


In [ ]:
from datetime import datetime, timezone, timedelta
# datetime = Python built-in module for working with dates and times
# timedelta = a duration (e.g., timedelta(hours=-5) = 'subtract 5 hours')

# Stage 3: PROCESSING — clean, transform, enrich (Silver layer)
# WHY: Raw data is rarely analysis-ready.
#   - Timestamps are in UTC (Universal Coordinated Time); reports use local time (EST = Eastern Standard Time)
#   - Customer info is nested; SQL and BI (Business Intelligence) tools expect flat rows
#   - Duration is not stored — must be calculated from start/end times
# Silver layer = quality-checked, conformed (standardised column names and types across all tables)

def process_call(raw):
    """Transform raw call event into clean, analysis-ready record."""

    # WHY: Parse timestamps before doing any time math.
    # fromisoformat = parse an ISO 8601 timestamp string into a Python datetime object
    # ISO 8601 = the international standard for date/time strings: YYYY-MM-DDTHH:MM:SS
    # .replace("Z", "+00:00") — "Z" means UTC in ISO format; Python needs "+00:00" to
    # recognise it as a timezone-aware datetime (otherwise math across timezones breaks)
    start_utc = datetime.fromisoformat(raw["start_time"].replace("Z", "+00:00"))
    end_utc   = datetime.fromisoformat(raw["end_time"].replace("Z", "+00:00"))

    # WHY convert UTC -> EST?
    # Real bug: a call at 23:30 UTC = March 10 becomes 18:30 EST = March 10. Fine.
    # But a call at 00:15 UTC March 11 = 19:15 EST March 10 — DIFFERENT date!
    # Without this conversion, that call lands in the wrong day's report.
    # EST = UTC - 5 hours (Eastern Standard Time, without daylight saving for simplicity)
    est_offset = timedelta(hours=-5)  # WHY -5: EST is 5 hours behind UTC
    start_est  = start_utc + est_offset

    # Flatten nested customer dict — SQL tables expect one value per column, not nested dicts
    customer = raw.get("customer", {})
    # .get("customer", {}) = safely get "customer" key; return empty dict if missing (avoids KeyError)

    # Calculate duration — WHY: source system only stores start/end; consumers need seconds
    # (end_utc - start_utc) returns a timedelta object; .total_seconds() converts to a float
    duration = (end_utc - start_utc).total_seconds()

    return {
        "call_id":           raw["call_id"],
        "dnis":              raw["dnis"],
        "caller_phone":      raw["caller_phone"],
        "start_time_utc":    start_utc.isoformat(),   # keep original UTC for audit
        "start_time_est":    start_est.isoformat(),   # local time for reporting
        "call_date":         start_est.strftime("%Y-%m-%d"),  # EST date used as report date
        "call_hour":         start_est.hour,           # hour of day in EST for time-of-day analysis
        "duration_seconds":  duration,
        "disposition":       raw["disposition"],
        # Flattened customer fields — each value becomes its own column
        "customer_first_name": customer.get("first_name"),
        "customer_last_name":  customer.get("last_name"),
        "customer_city":       customer.get("city"),
        "customer_state":      customer.get("state"),
    }

clean_record = process_call(raw_event)

print("Stage 3 — PROCESSING: Cleaned and transformed record")
print(f"  UTC start:  {raw_event['start_time']}")
print(f"  EST start:  {clean_record['start_time_est']}")
print(f"  Call date:  {clean_record['call_date']} (EST — this is the date reports use)")
print(f"  Call hour:  {clean_record['call_hour']} (2:30 PM EST — not 7:30 PM UTC!)")
print(f"  Duration:   {clean_record['duration_seconds']}s ({clean_record['duration_seconds']/60:.1f} min)")
print(f"  Customer:   {clean_record['customer_first_name']} {clean_record['customer_last_name']}, {clean_record['customer_city']}, {clean_record['customer_state']}")
print()
print("Notice: nested JSON is flattened, UTC is converted, duration is calculated.")
print("This is what a DE does — make raw data usable.")
# You should see: EST time 5 hours earlier than UTC, call_date = 2026-03-10, duration = 465s


In [ ]:
# Stage 4: SERVING — make it available for reports/analysis (Gold layer)
# WHY: Stakeholders (managers, analysts) need pre-aggregated, easy-to-query data.
# Gold layer = business-ready, optimised for consumption (BI tools, dashboards, SQL)
# In production: write to BigQuery (Google's cloud data warehouse),
# create materialised views, or power a dashboard in Looker/Tableau.
# Here: simulate with a plain Python aggregation — no installs required.

# Sample data: a batch of already-processed (Silver) calls
# WHY a list of dicts: in real life this is a Spark DataFrame or a SQL result set.
daily_calls = [
    {"campaign": "Campaign A", "disposition": "completed",  "duration": 465, "amount": 79.99},
    {"campaign": "Campaign A", "disposition": "completed",  "duration": 380, "amount": 119.98},
    {"campaign": "Campaign A", "disposition": "dropped",    "duration": 45,  "amount": 0},
    {"campaign": "Campaign A", "disposition": "completed",  "duration": 510, "amount": 79.99},
    {"campaign": "Campaign A", "disposition": "dropped",    "duration": 30,  "amount": 0},
    {"campaign": "Campaign B", "disposition": "completed",  "duration": 390, "amount": 49.95},
    {"campaign": "Campaign B", "disposition": "completed",  "duration": 420, "amount": 49.95},
    {"campaign": "Campaign B", "disposition": "transferred","duration": 200, "amount": 0},
    {"campaign": "Campaign B", "disposition": "dropped",    "duration": 15,  "amount": 0},
    {"campaign": "Campaign B", "disposition": "completed",  "duration": 350, "amount": 49.95},
]

# Build a simple campaign report — this is what the Gold layer produces
from collections import defaultdict
# defaultdict = a dict that auto-creates a default value for any new key
# WHY: avoids writing 'if key not in dict: dict[key] = 0' before every increment
# lambda = a tiny anonymous one-line function
# lambda: {"total": 0, ...} means "when a new key is accessed, create this dict as default"
stats = defaultdict(lambda: {"total": 0, "completed": 0, "revenue": 0.0, "total_duration": 0})

# Loop through every call and accumulate stats per campaign
for call in daily_calls:
    c = call["campaign"]           # WHY: group results by campaign name
    stats[c]["total"] += 1         # count every call regardless of outcome
    stats[c]["total_duration"] += call["duration"]
    if call["disposition"] == "completed":  # only completed calls generate revenue
        stats[c]["completed"] += 1
        stats[c]["revenue"] += call["amount"]

# Print the Gold-layer report
print("Stage 4 — SERVING: Daily Campaign Report (Gold Layer)")
print("=" * 75)
# f-string format spec: {value:<18} = left-align in 18 chars, {:>6} = right-align in 6 chars
print(f"{'Campaign':<18} {'Calls':>6} {'Converted':>10} {'Conv %':>8} {'Revenue':>10} {'Avg Dur':>8}")
print("-" * 75)
for campaign, s in stats.items():
    # WHY guard against division by zero: if total is 0 we'd get ZeroDivisionError
    conv_rate = (s["completed"] / s["total"] * 100) if s["total"] > 0 else 0
    avg_dur   = s["total_duration"] / s["total"] if s["total"] > 0 else 0
    print(f"{campaign:<18} {s['total']:>6} {s['completed']:>10} {conv_rate:>7.1f}% {s['revenue']:>9.2f} {avg_dur:>7.0f}s")
print("=" * 75)
print()
print("This report is what business stakeholders see.")
print("Your job: make sure the data is correct, timely, and reliable.")
# You should see: Campaign A = 3 conversions (60%), Campaign B = 3 (60%), different revenues


---

## 3. ETL vs. ELT Paradigms

**One-line answer:** ETL cleans data before storing it; ELT stores raw data first and cleans it inside the warehouse — most modern pipelines use ELT.

> Think of ETL like cooking a meal before boxing it for delivery — the box only contains finished food. ELT is like shipping raw ingredients to the restaurant and letting their kitchen prepare it — the raw ingredients are always there if you need to start over.


Two fundamental patterns for moving data. You'll use both in your career.

### ETL: Extract → Transform → Load

```
Source  ──→  Transform (outside the warehouse)  ──→  Load into warehouse
```

- Transform happens **before** the data reaches the warehouse
- The warehouse only sees clean data
- Traditional approach (Informatica, SSIS, custom scripts)
- You need compute power **outside** the warehouse (Spark cluster, Python server)

### ELT: Extract → Load → Transform

```
Source  ──→  Load raw into warehouse  ──→  Transform inside the warehouse
```

- Raw data lands in the warehouse first
- Transform happens **inside** the warehouse using SQL
- Modern approach (BigQuery, Snowflake, Databricks)
- The warehouse has enough compute to handle transformations

### Comparison

| Aspect | ETL | ELT |
|--------|-----|-----|
| **Transform happens** | Before loading (external compute) | After loading (warehouse compute) |
| **Raw data in warehouse?** | No — only clean data arrives | Yes — raw data lands first |
| **Compute** | External (Spark, Python, etc.) | Warehouse engine (BigQuery, Snowflake) |
| **Best for** | Complex transforms, legacy systems, multiple targets | Cloud warehouses with cheap compute |
| **Audit trail** | Harder — raw data may not be preserved | Easier — raw data is always in the warehouse |
| **Cost model** | Pay for external compute cluster | Pay for warehouse processing |
| **When to use** | PySpark transforms, ML pipelines, multi-system | SQL-heavy transforms, analyst self-service |

---
### Why Does This Work? — ETL vs. ELT Decision Guide

**The question you will get in every DE interview:** *"When do you use ETL vs. ELT?"*

| Situation | Choose | Reason |
|-----------|--------|--------|
| Complex logic SQL cannot express (ML feature engineering, regex parsing, custom dedup) | **ETL** | Python/Spark can do things SQL cannot |
| Multiple destination systems (same data goes to BigQuery AND a REST API AND a file) | **ETL** | Transform once in code, write to many targets |
| Cloud warehouse with cheap compute (BigQuery, Snowflake, Databricks) | **ELT** | Warehouse does the heavy lifting; raw data is preserved |
| Analysts need to self-serve and write their own transforms | **ELT** | They can write SQL against Bronze; no DE bottleneck |
| Regulatory audit trail required (healthcare, finance) | **ELT** | Raw data in Bronze = immutable record of what arrived |
| Legacy destination with no compute (old Oracle, flat-file system) | **ETL** | Destination cannot transform; data must arrive clean |

**One-sentence rule:** Use ELT when the warehouse is powerful enough and you want to preserve raw data. Use ETL when you need code-level transforms or cannot land raw data in the warehouse.

> **Interview tip:** Never say "I always use ETL" or "I always use ELT." Show you know the trade-offs. That is what separates a junior who memorises definitions from one who can make production decisions.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ETL = Extract, Transform, Load
# Pattern: clean/transform data BEFORE it reaches the warehouse
# WHY choose ETL: useful when transforms require Python/Spark logic that SQL cannot
# express (regex parsing, ML feature engineering, complex deduplication), or when
# the destination system expects clean data and has no compute for transforms.
# ─────────────────────────────────────────────────────────────────────────────

# Raw data from source system — messy on purpose (duplicates, nulls, mixed case)
raw_calls = [
    {"id": "call_001", "ts": "2026-03-10T19:30:00Z", "dur": "7m45s", "status": "COMPLETED"},
    {"id": "call_002", "ts": "2026-03-10T20:15:00Z", "dur": "3m10s", "status": "dropped"},
    {"id": "call_001", "ts": "2026-03-10T19:30:00Z", "dur": "7m45s", "status": "COMPLETED"},  # duplicate — same id!
    {"id": "call_003", "ts": "2026-03-10T21:00:00Z", "dur": None,    "status": "COMPLETED"},  # null duration
]

print("=== ETL: Extract, Transform, Load — Transform FIRST, then load ===")
print(f"\nRaw records received: {len(raw_calls)}")

# TRANSFORM (happens OUTSIDE the warehouse, in Python/Spark)

def parse_duration(dur_str):
    """Convert human-readable duration string '7m45s' to integer seconds.

    WHY a helper function: keeps the main loop clean and makes this logic reusable.
    WHY handle None: source systems sometimes omit fields for very short/dropped calls.
    """
    if dur_str is None:
        return 0  # default for nulls — avoids crashing on missing data
    # Split on 'm' to get ['7', '45s'], then strip 's' from the seconds part
    minutes = int(dur_str.split('m')[0])
    seconds = int(dur_str.split('m')[1].replace('s', ''))
    return minutes * 60 + seconds  # convert everything to seconds for consistent storage

# Step 1: Deduplicate — remove records with the same id
# WHY: duplicate records inflate call counts and revenue totals in reports
# set() = an unordered collection of UNIQUE values (like a list that rejects duplicates)
seen_ids = set()
deduped  = []
for call in raw_calls:
    if call["id"] not in seen_ids:   # first time we see this id -> keep it
        seen_ids.add(call["id"])
        deduped.append(call)
    # WHY no else: silently drop the duplicate — it's identical, nothing to keep

# Step 2: Clean and standardise field names and formats
# WHY standardise: downstream systems (BI tools, SQL queries) expect consistent names
clean_calls = []
for call in deduped:
    clean_calls.append({
        "call_id":          call["id"],              # rename abbreviated field
        "start_time":       call["ts"],              # rename abbreviated field
        "duration_seconds": parse_duration(call["dur"]),  # convert "7m45s" -> 465
        "disposition":      call["status"].lower(),  # normalise to lowercase (COMPLETED -> completed)
    })

print(f"After dedup: {len(clean_calls)} records (removed {len(raw_calls) - len(clean_calls)} duplicate)")
print(f"\nClean data ready to LOAD into warehouse:")
for c in clean_calls:
    print(f"  {c}")

print()
print("-> The warehouse only sees clean, deduplicated, standardised data.")
# You should see: 3 records (call_001, call_002, call_003), all lowercase dispositions
print("-> But: if the transform had a bug, the raw data may be gone — no way to re-derive.")
print("   This is why ELT (keeping raw data in the warehouse) is preferred in modern DE.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ELT = Extract, Load, Transform
# Pattern: dump raw data into the warehouse FIRST, transform it there using SQL
# WHY modern teams prefer ELT:
#   1. Raw data is always preserved — bugs in transform? Re-run it any time.
#   2. Cloud warehouses (BigQuery, Snowflake) have massive SQL compute — use it.
#   3. Analysts can self-serve by writing SQL against Silver/Gold tables directly.
# ─────────────────────────────────────────────────────────────────────────────

print("=== ELT: Extract, Load, Transform — Load raw FIRST, transform in warehouse ===")
print(f"\nRaw records: {len(raw_calls)}")

# EXTRACT + LOAD
import sqlite3
# sqlite3 = Python's built-in lightweight SQL database
# WHY sqlite3: simulates BigQuery or Snowflake without needing cloud credentials
# ":memory:" = create the database entirely in RAM — disappears when the program ends
conn   = sqlite3.connect(":memory:")
cursor = conn.cursor()  # cursor = the object that sends SQL commands to the database

# Create Bronze table — raw data, NO cleaning, exactly as received from the source
# WHY preserve raw: this is your audit trail and your safety net for debugging
cursor.execute("""
    CREATE TABLE bronze_calls (
        id     TEXT,   -- abbreviated field name kept intentionally (as-received)
        ts     TEXT,   -- timestamp as a string (not parsed yet)
        dur    TEXT,   -- duration as string '7m45s' (not converted yet)
        status TEXT    -- mixed case (COMPLETED, dropped) - not normalised yet
    )
""")

# Insert ALL raw records — duplicates, nulls, mixed case and all
# WHY: in ELT, Bronze is append-only. Everything lands here first.
for call in raw_calls:
    cursor.execute("INSERT INTO bronze_calls VALUES (?, ?, ?, ?)",
                   (call["id"], call["ts"], call["dur"], call["status"]))
    # ? placeholders = parameterised query — prevents SQL injection attacks
    # SQL injection = a security vulnerability where malicious input manipulates your query

print("Step 1 — LOADED raw data into bronze_calls (duplicates, nulls, and all):")
for row in cursor.execute("SELECT * FROM bronze_calls"):
    print(f"  {row}")

# TRANSFORM (happens INSIDE the warehouse using SQL)
# WHY SQL inside the warehouse: warehouse engines are optimised for this —
# they can transform billions of rows faster than Python loops can.
# DISTINCT = deduplication in SQL (keeps one row per unique combination of all columns)
# COALESCE = return the first non-NULL value from a list — handles the null dur field
# LOWER() = normalise text to lowercase — same as .lower() in Python
cursor.execute("""
    CREATE TABLE silver_calls AS
    SELECT DISTINCT
        id     AS call_id,              -- rename abbreviated column
        ts     AS start_time,           -- rename abbreviated column
        COALESCE(dur, '0m0s') AS duration_raw,  -- replace NULL with default '0m0s'
        LOWER(status) AS disposition    -- normalise case: 'COMPLETED' -> 'completed'
    FROM bronze_calls
""")
# CREATE TABLE ... AS SELECT (CTAS pattern) = fast way to materialise a transformed query as a table

print("\nStep 2 — TRANSFORMED inside warehouse (silver_calls):")
for row in cursor.execute("SELECT * FROM silver_calls"):
    print(f"  {row}")
# You should see: 3 rows (duplicate removed by DISTINCT), all lowercase dispositions

print()
print("-> Raw data is PRESERVED in bronze_calls. If transform has a bug, re-run it.")
print("-> This is why ELT is preferred in modern data engineering.")

conn.close()  # WHY close: releases the database connection and frees memory
              # Always close connections when done — critical habit in production code


### In Practice: Most Pipelines Blend Both

Most modern data pipelines use a hybrid approach:

- **Light ETL** on the way in: basic schema enforcement, format conversion, deduplication (PySpark)
- **Heavy ELT** in the warehouse: aggregations, window functions, business logic (BigQuery SQL)

This combination gives you the best of both worlds — complex transforms where you need code (PySpark), simple transforms where SQL is faster (BigQuery).

> **In this program:** You'll use PySpark for complex transforms (ETL-style) and BigQuery for analytical queries (ELT-style). You'll see both patterns in action starting in Module 06.

---

## 4. Batch vs. Real-Time Processing

**One-line answer:** Batch processes data on a schedule (hourly, daily); real-time processes it the moment it arrives.

> Batch is like doing all your laundry on Sunday — you collect it all week and wash everything at once. Real-time is like washing each item the moment you take it off — faster feedback, but more expensive to run.


| Aspect | Batch Processing | Real-Time (Stream) Processing |
|--------|-----------------|------------------------------|
| **When it runs** | On a schedule (hourly, daily, weekly) | Continuously, as events arrive |
| **Data volume** | Large chunks at once | One event at a time |
| **Latency** | Minutes to hours | Milliseconds to seconds |
| **Tools** | Spark, Airflow, dbt | Kafka, Flink, Spark Streaming |
| **Complexity** | Lower — well-understood patterns | Higher — ordering, exactly-once, state |
| **Cost** | Lower — compute only when running | Higher — compute always running |
| **Example** | "Generate yesterday's report at 6 AM" | "Alert me the moment a payment fails" |

### Where Each Fits

| Use Case | Type | Why |
|----------|------|-----|
| Daily campaign performance report | **Batch** | Stakeholders check it every morning — doesn't need to be real-time |
| Refresh reporting tables | **Batch** | Can run on a schedule (every 15 min, hourly, daily) |
| Payment failure alert | **Real-time** | Need to know immediately if the payment gateway is down |
| Fraud detection | **Real-time** | Can't wait for a batch to catch a stolen card |
| Monthly revenue reconciliation | **Batch** | Runs once a month, processes millions of records |
| Live agent availability dashboard | **Real-time** | Supervisors need live view of who's on calls |

> **In this program:** We focus on **batch processing**. That's where 90% of DE work happens, and it's where Jr. DEs start. Real-time (Kafka, Flink) is a specialization you'll add later in your career.

### Micro-Batch: The Middle Ground

Many production systems run batch on a tight schedule (every 5-15 minutes). This is called **micro-batch** — not truly real-time, but frequent enough for most reporting needs. It's simpler to build and operate than a full streaming pipeline.

```
True Batch:    |──── 24 hrs ────|──── 24 hrs ────|  (daily runs)
Micro-Batch:   |--15m--|--15m--|--15m--|--15m--|  (frequent runs)
Real-Time:     ───────────────────────────  (continuous)
```

---
### Why Does This Work? — Batch vs. Real-Time Decision Guide

**The question:** *"When do you use batch processing vs. real-time streaming?"*

| Situation | Choose | Reason |
|-----------|--------|--------|
| Daily report that stakeholders check every morning | **Batch** | Data only needs to be fresh once a day — no reason to pay for streaming |
| Payment failure alert | **Real-time** | Every second of delay = more failed transactions |
| Monthly revenue reconciliation | **Batch** | Runs once, processes millions of rows — perfect for Spark |
| Fraud detection on credit card swipes | **Real-time** | Must catch fraud before the transaction clears |
| Refreshing a campaign performance dashboard | **Micro-batch** (every 5-15 min) | Frequent enough for managers; far cheaper than true streaming |
| Live agent availability board | **Real-time** | Supervisors need current state, not 1-hour-old state |

**Cost reality:** A real-time Kafka + Flink pipeline costs 10-50x more to build and operate than a batch Spark job. Have a compelling business reason before choosing streaming.

**Micro-batch is your friend:** Most "real-time" reporting needs are satisfied by running a batch job every 5 minutes. Apache Airflow can schedule at that frequency — far simpler than true streaming.

> **Interview tip:** Demonstrate that you think about cost and complexity, not just technical capability. Saying "I'd use Kafka because it's real-time" without justifying the cost will raise a red flag.

---

## 5. Modern Data Stack Overview

**One-line answer:** The modern data stack separates storage from compute, treats pipelines as code, and organizes data in Bronze/Silver/Gold layers.

> **Terms you'll hear today** (don't memorize — just recognize):
> - **dbt** = SQL transformation tool
> - **Fivetran/Airbyte** = automated data connectors
> - **Great Expectations** = data quality testing
> - **Medallion architecture** = Bronze → Silver → Gold data layers

The tools and infrastructure that modern data teams use. This has changed dramatically in the last 5 years.

### Why the Shift?

Three forces drove the change from legacy to modern data stacks:

1. **Cloud economics:** Separation of compute and storage. Pay for what you use instead of buying servers.

> Think of it like renting a U-Haul: in the old world, you bought a truck (compute) that came with a permanent trailer (storage). Now you rent the truck only when you need to move, and your stuff sits in a storage unit (S3/GCS) that you pay for by the month.

2. **Data volume:** Terabytes to petabytes — traditional tools (SSIS, stored procedures) couldn't keep up.
3. **Code-based pipelines:** Version-controlled, testable, reviewable — data engineering became software engineering.

### Then vs. Now

| Layer | Legacy (2015) | Modern (2025+) |
|-------|--------------|----------------|
| **Ingestion** | Informatica, SSIS, custom scripts | Fivetran, Airbyte (tools that automatically sync data from sources like Salesforce or Stripe into your warehouse), custom PySpark |
| **Storage** | On-prem SQL Server, Oracle | Cloud: BigQuery, Snowflake, Databricks, S3/GCS |
| **Processing** | Stored procedures, SSIS packages | PySpark, dbt (a SQL transformation tool that lets you version-control and test your warehouse queries), SQL in warehouse |
| **Table Format** | Regular tables | Delta Lake, Apache Iceberg (ACID (Atomicity, Consistency, Isolation, Durability — guarantees that database transactions are reliable) on data lakes) |
| **Orchestration** | Windows Task Scheduler, cron, SSIS | Apache Airflow, Dagster, Prefect |
| **CI/CD** | Manual deployment | GitHub Actions, automated testing |
| **Quality** | Manual checks, hope | Great Expectations (a Python framework for writing data quality checks), dbt tests, automated validation |
| **Monitoring** | "Did anyone complain?" | Alerts, dashboards, SLAs (Service Level Agreements — promises about uptime, freshness, or data quality) |

### The Medallion Architecture

A key pattern in the modern data stack is **layered storage** — often called the medallion architecture:

> Bronze is your walk-in fridge — raw ingredients, untouched, exactly as delivered. Silver is the prep station — washed, chopped, portioned. Gold is the finished plate that goes to the customer.

```
BRONZE          →  SILVER            →  GOLD
(raw, as-is)       (cleaned, joined)     (business-ready, aggregated)
```

| Layer | Purpose | Key Property |
|-------|---------|-------------|
| **Bronze** | Store raw data exactly as received | Append-only, immutable (you never edit raw data — only add new records), full audit trail |
| **Silver** | Clean, deduplicate, validate, join | Quality-checked, conformed schemas (standardized column names, types, and formats across all tables) |
| **Gold** | Business-ready marts for reporting | Pre-aggregated, optimized for queries |

Why 3 layers? **Single responsibility.** Bronze never changes (audit trail). Silver is where bugs get caught. Gold is what business users see. If Gold has bad data, fix Silver and re-derive.

### The Stack You'll Learn in This Program

| Layer | Tool | Module |
|-------|------|--------|
| **Language** | Python, SQL | M03, M06 |
| **Processing** | PySpark (Apache Spark) | M06, M07 |
| **Cloud Platform** | GCP (BigQuery, Dataproc, Cloud Storage) | M08 |
| **Table Format** | Delta Lake, Apache Iceberg | M09, M10 |
| **Orchestration** | Apache Airflow | M11 |
| **CI/CD** | GitHub Actions | M12 |
| **Version Control** | Git | M02 |
| **Data Modeling** | Star schema, dimensional modeling | M04 |
| **Legacy Context** | Hadoop, Hive | M05 |

> **Why GCP?** BigQuery is the fastest-growing cloud data warehouse. Dataproc gives you managed Spark. Cloud Composer gives you managed Airflow. One ecosystem, one billing console. The concepts transfer to AWS and Azure.

---

## Key Takeaways

1. **Data engineers build the infrastructure** that moves data from where it's generated to where it's useful — pipelines, storage, processing, serving.
2. **The data lifecycle** is generation → storage → processing → serving. You own stages 2-4.
3. **ETL transforms before loading** (external compute); **ELT loads first, transforms in the warehouse** (warehouse compute). Modern pipelines blend both.
4. **Batch processing** handles the vast majority of DE work — scheduled runs that process data in chunks. Real-time is a specialization you'll add later.
5. **The modern data stack** has shifted from on-prem tools (SSIS, stored procedures, Task Scheduler) to cloud-native tools (PySpark, BigQuery, Airflow, GitHub Actions).
6. **The medallion architecture** (Bronze → Silver → Gold) separates raw storage from clean data from business-ready marts.

---

## Discussion Questions

1. **Where have you encountered data quality issues** in your previous work or studies? What happened?
2. **Can you think of a case** where batch processing wouldn't be fast enough? What would you need instead?
3. **Why has the data stack shifted from on-prem to cloud?** What are the advantages? Any disadvantages?
4. **When would you choose ETL over ELT?** Think about a scenario where each is the better fit.

---## DE Interview QuestionsThese are the questions you will be asked in a Jr. Data Engineer interview. Know them cold.| # | Question | Key Points ||---|----------|------------|| 1 | **What does a Data Engineer do?** | Builds the infrastructure that moves data from where it's generated to where it's useful. Pipelines, storage, processing, serving. You don't analyze data — you make it analyzable. || 2 | **ETL vs ELT — when do you use each?** | ETL: transform before loading (external compute, PySpark). ELT: load raw first, transform in warehouse (BigQuery, Snowflake). Modern pipelines blend both. ELT preferred when warehouse has cheap compute. || 3 | **When would you use batch vs streaming?** | Batch (90% of DE work): scheduled runs — daily reports, data refreshes. Streaming: fraud detection, live dashboards, payment alerts. Start with batch. Add streaming only when latency matters. || 4 | **Data lake vs data warehouse?** | Lake: raw files (S3/GCS), any format, schema-on-read, cheap storage. Warehouse: structured tables (BigQuery/Snowflake), schema-on-write, optimized for SQL queries. Modern stack uses both (lakehouse). || 5 | **What is a data pipeline?** | Code that moves data from A to B with transformations in between. Read → transform → write. Can be 3 lines of Python or 10,000 lines of PySpark on 100 machines. The shape is always the same. || 6 | **What is the medallion architecture?** | Bronze (raw, as-is) → Silver (cleaned, validated) → Gold (aggregated, business-ready). Single responsibility: Bronze never changes, Silver catches bugs, Gold serves reports. || 7 | **What does "idempotent" mean in DE?** | Running a pipeline twice produces the same result. If the daily job runs twice, you don't get duplicate rows. Critical for reliability. |> **Preparation tip:** Don't memorize definitions. Explain each concept using the call center example from this notebook. Interviewers want to see you can connect concepts to real work.

---

## Homework (Before Next Session)

1. **Verify your setup from M00** — run the verification checklist:
   ```bash
   python3 --version          # Python 3.10+
   pip3 --version             # pip (package manager)
   git --version              # Git 2.x+
   code --version             # VS Code
   ```
   If anything is missing, refer to the Setup & Installation section in the M00 notebook.
2. **Configure Git identity** (if you haven't):
   ```bash
   git config --global user.name "Your Name"
   git config --global user.email "your.email@example.com"
   ```
3. **Review SQL basics** if rusty:
   - [SQLBolt](https://sqlbolt.com/) — Lessons 1-6 (30 min)
   - Know: SELECT, WHERE, JOIN, GROUP BY, ORDER BY
4. **Reflect on today's discussion questions** — bring one real-world example of a data quality issue you've seen or read about.

---

**Next session:** M02 (Git & Linux) + start M03 (Advanced SQL)